# 01 — State-vector simulator: from-scratch vs Qiskit

This is the **Phase 1 acceptance gate** in notebook form. We build a few canonical small circuits with our hand-rolled `qec.statevec` simulator and cross-check the resulting state vectors against `qiskit.quantum_info.Statevector`.

If everything matches to 1e-12, our reshape-based simulator is correct (and we'll re-use it as a cross-check oracle for Phases 2–5).

**Read first:** `notes/00-linear-algebra.md`, `notes/01-single-qubit.md`, `notes/02-multi-qubit.md`.

---


In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

from qec import statevec as sv

ATOL = 1e-12
rng = np.random.default_rng(2026)


## 1. Hadamard creates a uniform superposition

`H |0> = (|0> + |1>) / sqrt(2)`. Both probabilities should be exactly 0.5.


In [ ]:
psi = sv.zero_state(1)
psi = sv.apply_1q(psi, sv.H, 0)
print("amplitudes:", np.round(psi, 6))
print("probabilities:", np.round(sv.probabilities(psi), 6))


## 2. Bell state, our simulator vs Qiskit

The standard recipe: `H` on qubit 0, then `CNOT(0 -> 1)`. Result should be
`(|00> + |11>) / sqrt(2)`.


In [ ]:
# our simulator
psi = sv.zero_state(2)
psi = sv.apply_1q(psi, sv.H, 0)
psi = sv.cnot(psi, control=0, target=1)

# Qiskit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qiskit_psi = np.asarray(Statevector.from_instruction(qc).data, dtype=complex)

print("qec.statevec:", np.round(psi, 6))
print("qiskit:      ", np.round(qiskit_psi, 6))
print("max abs diff:", np.max(np.abs(psi - qiskit_psi)))
assert np.allclose(psi, qiskit_psi, atol=ATOL)
print("OK")


## 3. GHZ on 5 qubits

`(|00000> + |11111>) / sqrt(2)`. Build it both ways and confirm equality.


In [ ]:
n = 5
# our simulator
psi = sv.zero_state(n)
psi = sv.apply_1q(psi, sv.H, 0)
for q in range(1, n):
    psi = sv.cnot(psi, control=0, target=q)

# Qiskit
qc = QuantumCircuit(n)
qc.h(0)
for q in range(1, n):
    qc.cx(0, q)
qiskit_psi = np.asarray(Statevector.from_instruction(qc).data, dtype=complex)

print("non-zero amplitudes (qec):", np.round(psi[psi != 0], 6))
print("max abs diff:             ", np.max(np.abs(psi - qiskit_psi)))
assert np.allclose(psi, qiskit_psi, atol=ATOL)
print("OK")


## 4. Random Clifford circuit, 6 qubits, 80 gates

The real stress test: a random Clifford circuit. If our simulator handles this, it handles everything Phase 1 needs.


In [ ]:
n, depth = 6, 80
qc = QuantumCircuit(n)
psi = sv.zero_state(n)

for _ in range(depth):
    kind = rng.integers(0, 4)
    q = int(rng.integers(0, n))
    if kind == 0:
        qc.h(q);          psi = sv.apply_1q(psi, sv.H, q)
    elif kind == 1:
        qc.s(q);          psi = sv.apply_1q(psi, sv.S, q)
    elif kind == 2:
        qc.x(q);          psi = sv.apply_1q(psi, sv.X, q)
    else:
        q2 = int(rng.integers(0, n))
        while q2 == q:
            q2 = int(rng.integers(0, n))
        qc.cx(q, q2);     psi = sv.cnot(psi, q, q2)

qiskit_psi = np.asarray(Statevector.from_instruction(qc).data, dtype=complex)
print("max abs diff over 2^6 amplitudes:", np.max(np.abs(psi - qiskit_psi)))
assert np.allclose(psi, qiskit_psi, atol=ATOL)
print("OK")


## 5. Quantum teleportation

Three qubits: q0 holds the unknown state `|psi>`, (q1, q2) are a Bell pair shared between Alice and Bob.

1. CNOT(q0 -> q1), then H on q0.
2. Measure q0 and q1 in the Z basis.
3. Apply X^{m1} Z^{m0} to q2.

After step 3, q2 should be in state `|psi>` (and q0, q1 are in some product state).


In [ ]:
# Random unknown 1-qubit state
alpha = rng.normal() + 1j * rng.normal()
beta  = rng.normal() + 1j * rng.normal()
norm  = np.sqrt(np.abs(alpha)**2 + np.abs(beta)**2)
alpha, beta = alpha / norm, beta / norm
print("unknown state (alpha, beta) =", np.round(alpha, 4), np.round(beta, 4))

# Build a 3-qubit register: q0 = unknown, q1+q2 = Bell pair
psi = np.zeros(2**3, dtype=complex)
# qubit ordering: index = b2 b1 b0  (q0 is least significant)
psi[0b000] = alpha            # q2=0, q1=0, q0=0
psi[0b001] = beta             # q2=0, q1=0, q0=1
# Then create Bell pair on (q1, q2):
psi = sv.apply_1q(psi, sv.H, 1)
psi = sv.cnot(psi, control=1, target=2)

# Teleport: CNOT(q0->q1), H on q0, then measure q0 and q1
psi = sv.cnot(psi, control=0, target=1)
psi = sv.apply_1q(psi, sv.H, 0)
m0, psi = sv.measure_qubit(psi, 0, rng=rng)
m1, psi = sv.measure_qubit(psi, 1, rng=rng)
print("measurement outcomes (m0, m1) =", m0, m1)

# Conditional corrections on q2
if m1 == 1:
    psi = sv.apply_1q(psi, sv.X, 2)
if m0 == 1:
    psi = sv.apply_1q(psi, sv.Z, 2)

# Project out the (now classical) q0 and q1 — they're in basis state |m1 m0> for sure.
# Reduced amplitudes on q2:
idx0 = (m1 << 1) | m0                              # q1 q0
amp_q2_0 = psi[(0 << 2) | idx0]                    # q2=0
amp_q2_1 = psi[(1 << 2) | idx0]                    # q2=1
print("q2 amplitudes after teleport:", np.round(amp_q2_0, 4), np.round(amp_q2_1, 4))
print("expected (alpha, beta):       ", np.round(alpha, 4), np.round(beta, 4))

# Up to a global phase, q2 should equal the unknown state.
overlap = np.abs(np.conj(alpha) * amp_q2_0 + np.conj(beta) * amp_q2_1)
print("|<psi|q2>| =", round(overlap, 12), " (should be 1.0)")
assert np.isclose(overlap, 1.0, atol=1e-10)
print("OK")


## What this notebook gates

If every `assert` above passed, then for the rest of Phase 1 onwards we can **trust `qec.statevec` as an oracle**: any new code we write (Pauli class, stabilizer simulator, repetition code, …) can be cross-checked against it on small circuits without spinning up Qiskit every time.

Next: Phase 2 — noise channels (`notes/04-noise-channels.md`, not yet written).
